# PyTorch 기반 손실함수 계산식 이해 실습

이 노트북은 딥러닝에서 자주 사용하는 손실함수(Loss Function)를 **공식 계산 방식**과 **PyTorch 내장 함수 계산 결과**를 비교하면서 이해하기 위한 실습 자료입니다.

포함된 손실함수는 다음과 같습니다.

1. MSELoss: 평균제곱오차
2. L1Loss: 평균절대오차
3. HuberLoss: MSE와 MAE의 절충형 손실함수
4. BCELoss: 이진 교차 엔트로피
5. BCEWithLogitsLoss: Sigmoid + BCE 결합 손실함수
6. CrossEntropyLoss: 다중 클래스 분류 손실함수
7. NLLLoss: Negative Log Likelihood Loss
8. KLDivLoss: 두 확률분포 차이 계산 손실함수

각 코드 구문에는 초보자도 이해할 수 있도록 상세 주석을 작성했습니다.


## 1. 기본 라이브러리 불러오기

PyTorch를 사용하여 텐서 계산과 손실함수 계산을 수행합니다.

In [ ]:
# torch는 PyTorch의 핵심 라이브러리입니다.
# 텐서 생성, 수학 연산, 신경망 구성, 손실함수 계산 등에 사용됩니다.
import torch

# torch.nn은 신경망 계층, 활성화 함수, 손실함수 등을 제공하는 모듈입니다.
import torch.nn as nn

# torch.nn.functional은 함수형 방식의 신경망 연산을 제공하는 모듈입니다.
# 예를 들어 sigmoid, softmax, log_softmax 등을 직접 호출할 수 있습니다.
import torch.nn.functional as F

# 출력 결과를 보기 좋게 하기 위해 소수점 출력 자릿수를 설정합니다.
# precision=4는 소수점 이하 4자리까지 출력하겠다는 의미입니다.
torch.set_printoptions(precision=4)

# PyTorch 버전을 출력하여 현재 실행 환경을 확인합니다.
print("PyTorch version:", torch.__version__)

## 2. MSELoss: 평균제곱오차

MSE(Mean Squared Error)는 실제값과 예측값의 차이를 제곱한 뒤 평균을 계산하는 손실함수입니다.

공식은 다음과 같습니다.

$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2
$$

여기서 `y`는 실제값이고, `ŷ`는 예측값입니다.

MSE는 회귀 문제에서 많이 사용됩니다.

In [ ]:
# 실제 정답값을 텐서로 생성합니다.
# 예를 들어 실제 집값, 실제 온도, 실제 매출과 같은 연속형 숫자라고 생각할 수 있습니다.
y_true = torch.tensor([3.0, 5.0, 7.0])

# 모델이 예측한 값을 텐서로 생성합니다.
# 실제값과 약간 다르게 예측한 상황입니다.
y_pred = torch.tensor([2.5, 5.5, 8.0])

# 실제값과 예측값의 차이를 계산합니다.
# 결과: [3.0-2.5, 5.0-5.5, 7.0-8.0]
error = y_true - y_pred

# 오차를 제곱합니다.
# 제곱을 하면 음수 오차도 양수로 바뀌고, 큰 오차에 더 큰 벌점이 부여됩니다.
squared_error = error ** 2

# 제곱 오차들의 평균을 계산합니다.
# 이것이 MSE 공식에 따른 직접 계산 결과입니다.
manual_mse = squared_error.mean()

# PyTorch에서 제공하는 MSELoss 객체를 생성합니다.
# nn.MSELoss()는 평균제곱오차를 자동으로 계산합니다.
mse_loss = nn.MSELoss()

# PyTorch 내장 손실함수를 사용하여 MSE를 계산합니다.
torch_mse = mse_loss(y_pred, y_true)

# 계산 과정을 출력합니다.
print("실제값:", y_true)
print("예측값:", y_pred)
print("오차:", error)
print("제곱 오차:", squared_error)
print("직접 계산한 MSE:", manual_mse.item())
print("PyTorch MSELoss:", torch_mse.item())

## 3. L1Loss: 평균절대오차

L1Loss는 MAE(Mean Absolute Error)라고도 부릅니다.

실제값과 예측값의 차이에 절댓값을 씌운 뒤 평균을 계산합니다.

$$
MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|
$$

MAE는 MSE보다 이상치에 덜 민감합니다.

In [ ]:
# 실제 정답값을 생성합니다.
y_true = torch.tensor([3.0, 5.0, 7.0])

# 모델 예측값을 생성합니다.
y_pred = torch.tensor([2.5, 5.5, 8.0])

# 실제값과 예측값의 차이를 계산합니다.
error = y_true - y_pred

# 오차의 절댓값을 계산합니다.
# torch.abs()는 음수를 양수로 바꾸고, 양수는 그대로 유지합니다.
absolute_error = torch.abs(error)

# 절댓값 오차들의 평균을 계산합니다.
# 이것이 MAE 공식에 따른 직접 계산 결과입니다.
manual_mae = absolute_error.mean()

# PyTorch에서 제공하는 L1Loss 객체를 생성합니다.
# PyTorch에서는 MAE를 nn.L1Loss()라는 이름으로 제공합니다.
l1_loss = nn.L1Loss()

# PyTorch 내장 손실함수로 MAE를 계산합니다.
torch_mae = l1_loss(y_pred, y_true)

# 계산 결과를 출력합니다.
print("실제값:", y_true)
print("예측값:", y_pred)
print("오차:", error)
print("절댓값 오차:", absolute_error)
print("직접 계산한 MAE:", manual_mae.item())
print("PyTorch L1Loss:", torch_mae.item())

## 4. HuberLoss

HuberLoss는 오차가 작을 때는 MSE처럼 제곱 오차를 사용하고, 오차가 클 때는 MAE처럼 절댓값 오차를 사용합니다.

$$
L_{\delta}(a)=
\begin{cases}
\frac{1}{2}a^2, & |a| \leq \delta \\
\delta(|a|-\frac{1}{2}\delta), & |a| > \delta
\end{cases}
$$

여기서 `a = y - ŷ`입니다.

이상치가 있는 회귀 문제에서 유용합니다.

In [ ]:
# 실제 정답값을 생성합니다.
y_true = torch.tensor([1.0, 2.0, 10.0])

# 모델 예측값을 생성합니다.
# 마지막 값은 실제값과 차이가 큰 이상치 상황을 만들기 위해 0.0으로 설정했습니다.
y_pred = torch.tensor([1.5, 2.5, 0.0])

# HuberLoss에서 기준이 되는 delta 값을 설정합니다.
# 오차의 절댓값이 delta 이하이면 제곱 오차 방식, delta보다 크면 절댓값 오차 방식을 사용합니다.
delta = 1.0

# 실제값과 예측값의 차이를 계산합니다.
error = y_true - y_pred

# 오차의 절댓값을 계산합니다.
abs_error = torch.abs(error)

# torch.where는 조건에 따라 서로 다른 계산식을 적용할 수 있게 해줍니다.
# abs_error <= delta 이면 0.5 * error^2를 사용합니다.
# abs_error > delta 이면 delta * (abs_error - 0.5 * delta)를 사용합니다.
manual_huber_each = torch.where(
    abs_error <= delta,
    0.5 * error ** 2,
    delta * (abs_error - 0.5 * delta)
)

# 각 데이터별 HuberLoss 값을 평균냅니다.
manual_huber = manual_huber_each.mean()

# PyTorch의 HuberLoss 객체를 생성합니다.
# delta=1.0은 직접 계산에서 사용한 delta와 동일하게 맞춘 것입니다.
huber_loss = nn.HuberLoss(delta=delta)

# PyTorch 내장 함수로 HuberLoss를 계산합니다.
torch_huber = huber_loss(y_pred, y_true)

# 결과를 출력합니다.
print("실제값:", y_true)
print("예측값:", y_pred)
print("오차:", error)
print("오차 절댓값:", abs_error)
print("데이터별 HuberLoss:", manual_huber_each)
print("직접 계산한 HuberLoss:", manual_huber.item())
print("PyTorch HuberLoss:", torch_huber.item())

## 5. BCELoss: 이진 교차 엔트로피

BCELoss(Binary Cross Entropy Loss)는 이진 분류 문제에서 사용합니다.

정답은 0 또는 1이고, 예측값은 0과 1 사이의 확률값이어야 합니다.

$$
BCE = -\frac{1}{n}\sum_{i=1}^{n}
[y_i\log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]
$$

정답이 1일 때 예측확률이 1에 가까우면 손실이 작고, 정답이 0일 때 예측확률이 0에 가까우면 손실이 작습니다.

In [ ]:
# 실제 정답 라벨을 생성합니다.
# 1은 양성 클래스, 0은 음성 클래스를 의미합니다.
y_true = torch.tensor([1.0, 0.0, 1.0, 0.0])

# 모델이 예측한 확률값을 생성합니다.
# BCELoss에 들어가는 예측값은 반드시 0과 1 사이의 확률값이어야 합니다.
y_pred_prob = torch.tensor([0.9, 0.2, 0.8, 0.1])

# 로그 계산에서 log(0)이 발생하면 무한대 문제가 생길 수 있습니다.
# 아주 작은 값 epsilon을 사용하여 수치적으로 안전하게 계산합니다.
epsilon = 1e-7

# 예측 확률값이 정확히 0 또는 1이 되지 않도록 범위를 제한합니다.
# clamp는 지정한 최소값과 최대값 사이로 값을 잘라주는 함수입니다.
y_pred_safe = torch.clamp(y_pred_prob, epsilon, 1 - epsilon)

# BCE 공식을 직접 구현합니다.
# 정답이 1인 경우: -log(예측확률)
# 정답이 0인 경우: -log(1 - 예측확률)
manual_bce_each = -(
    y_true * torch.log(y_pred_safe) +
    (1 - y_true) * torch.log(1 - y_pred_safe)
)

# 데이터별 BCE 값을 평균냅니다.
manual_bce = manual_bce_each.mean()

# PyTorch의 BCELoss 객체를 생성합니다.
bce_loss = nn.BCELoss()

# PyTorch 내장 함수로 BCE를 계산합니다.
torch_bce = bce_loss(y_pred_prob, y_true)

# 결과를 출력합니다.
print("실제 라벨:", y_true)
print("예측 확률:", y_pred_prob)
print("데이터별 BCE:", manual_bce_each)
print("직접 계산한 BCE:", manual_bce.item())
print("PyTorch BCELoss:", torch_bce.item())

## 6. BCEWithLogitsLoss: Sigmoid + BCE

BCEWithLogitsLoss는 모델 출력값인 로짓(logit)에 Sigmoid를 적용한 뒤 BCE를 계산합니다.

즉, 다음 두 단계를 한 번에 처리합니다.

$$
\text{logit} \rightarrow \sigma(\text{logit}) \rightarrow BCE
$$

PyTorch에서는 이진 분류에서 `Sigmoid + BCELoss`보다 `BCEWithLogitsLoss`를 더 권장합니다.

이유는 수치적으로 더 안정적이기 때문입니다.

In [ ]:
# 실제 정답 라벨을 생성합니다.
y_true = torch.tensor([1.0, 0.0, 1.0, 0.0])

# 모델이 출력한 로짓값을 생성합니다.
# 로짓은 Sigmoid를 적용하기 전의 원시 점수입니다.
# 로짓은 0과 1 사이일 필요가 없고, 음수나 큰 양수도 가능합니다.
logits = torch.tensor([2.2, -1.4, 1.5, -2.0])

# 로짓에 Sigmoid를 적용하여 0과 1 사이의 확률값으로 변환합니다.
prob = torch.sigmoid(logits)

# Sigmoid로 변환된 확률값을 사용해 BCE 공식을 직접 계산합니다.
manual_bce_logits_each = -(
    y_true * torch.log(prob) +
    (1 - y_true) * torch.log(1 - prob)
)

# 데이터별 BCE 값을 평균냅니다.
manual_bce_logits = manual_bce_logits_each.mean()

# PyTorch의 BCEWithLogitsLoss 객체를 생성합니다.
# 이 함수는 내부적으로 Sigmoid와 BCE를 함께 계산합니다.
bce_logits_loss = nn.BCEWithLogitsLoss()

# 로짓값과 실제 라벨을 넣어 손실값을 계산합니다.
torch_bce_logits = bce_logits_loss(logits, y_true)

# 결과를 출력합니다.
print("실제 라벨:", y_true)
print("로짓값:", logits)
print("Sigmoid 적용 후 확률:", prob)
print("데이터별 BCE:", manual_bce_logits_each)
print("직접 계산한 Sigmoid + BCE:", manual_bce_logits.item())
print("PyTorch BCEWithLogitsLoss:", torch_bce_logits.item())

## 7. CrossEntropyLoss: 다중 클래스 분류 손실함수

CrossEntropyLoss는 여러 개의 클래스 중 하나를 고르는 다중 분류 문제에서 사용합니다.

PyTorch의 `CrossEntropyLoss`는 내부적으로 다음 두 과정을 수행합니다.

1. Softmax 계산
2. 정답 클래스에 대한 Negative Log Likelihood 계산

공식은 다음과 같습니다.

$$
CE = -\log(P(\text{정답 클래스}))
$$

여기서 `P(정답 클래스)`는 Softmax를 적용한 뒤 정답 클래스에 해당하는 확률입니다.

주의할 점은 PyTorch의 `CrossEntropyLoss`에는 One-Hot Encoding 정답이 아니라 클래스 번호를 넣는 것이 일반적입니다.

In [ ]:
# 모델이 출력한 로짓값을 생성합니다.
# 행은 데이터 샘플이고, 열은 클래스 점수입니다.
# 예를 들어 3개 샘플, 4개 클래스 분류 문제입니다.
logits = torch.tensor([
    [2.0, 1.0, 0.1, 0.0],
    [0.5, 2.5, 0.3, 0.2],
    [0.1, 0.2, 3.0, 0.4]
])

# 실제 정답 클래스 번호를 생성합니다.
# 첫 번째 샘플의 정답은 0번 클래스
# 두 번째 샘플의 정답은 1번 클래스
# 세 번째 샘플의 정답은 2번 클래스입니다.
target = torch.tensor([0, 1, 2])

# Softmax를 적용하여 각 클래스 점수를 확률로 변환합니다.
# dim=1은 각 행, 즉 각 샘플마다 클래스 방향으로 확률합이 1이 되도록 계산한다는 의미입니다.
softmax_prob = F.softmax(logits, dim=1)

# 각 샘플에서 정답 클래스에 해당하는 확률만 선택합니다.
# torch.arange(len(target))은 [0, 1, 2]를 만들어 각 샘플 행을 선택하게 합니다.
correct_class_prob = softmax_prob[torch.arange(len(target)), target]

# Cross Entropy 공식에 따라 정답 클래스 확률에 -log를 적용합니다.
manual_ce_each = -torch.log(correct_class_prob)

# 각 샘플의 Cross Entropy 값을 평균냅니다.
manual_ce = manual_ce_each.mean()

# PyTorch의 CrossEntropyLoss 객체를 생성합니다.
cross_entropy_loss = nn.CrossEntropyLoss()

# PyTorch 내장 함수로 Cross Entropy를 계산합니다.
# 입력은 Softmax 전 로짓값이고, 정답은 클래스 번호입니다.
torch_ce = cross_entropy_loss(logits, target)

# 결과를 출력합니다.
print("로짓값:")
print(logits)
print("\nSoftmax 확률:")
print(softmax_prob)
print("\n정답 클래스 번호:", target)
print("정답 클래스 확률:", correct_class_prob)
print("데이터별 CrossEntropy:", manual_ce_each)
print("직접 계산한 CrossEntropy:", manual_ce.item())
print("PyTorch CrossEntropyLoss:", torch_ce.item())

## 8. NLLLoss: Negative Log Likelihood Loss

NLLLoss는 로그 확률값을 입력으로 받아 정답 클래스의 음의 로그 가능도를 계산합니다.

공식은 다음과 같습니다.

$$
NLL = -\log(P(\text{정답 클래스}))
$$

`CrossEntropyLoss`는 내부적으로 `LogSoftmax + NLLLoss`와 거의 같은 역할을 합니다.

In [ ]:
# 모델이 출력한 로짓값을 생성합니다.
logits = torch.tensor([
    [2.0, 1.0, 0.1, 0.0],
    [0.5, 2.5, 0.3, 0.2],
    [0.1, 0.2, 3.0, 0.4]
])

# 실제 정답 클래스 번호를 생성합니다.
target = torch.tensor([0, 1, 2])

# LogSoftmax를 적용하여 로짓값을 로그 확률값으로 변환합니다.
# NLLLoss는 일반 확률이 아니라 로그 확률을 입력으로 받습니다.
log_prob = F.log_softmax(logits, dim=1)

# 각 샘플에서 정답 클래스에 해당하는 로그 확률만 선택합니다.
correct_log_prob = log_prob[torch.arange(len(target)), target]

# NLLLoss 공식에 따라 정답 클래스 로그 확률에 음수 부호를 붙입니다.
manual_nll_each = -correct_log_prob

# 데이터별 NLL 값을 평균냅니다.
manual_nll = manual_nll_each.mean()

# PyTorch의 NLLLoss 객체를 생성합니다.
nll_loss = nn.NLLLoss()

# PyTorch 내장 함수로 NLLLoss를 계산합니다.
torch_nll = nll_loss(log_prob, target)

# 결과를 출력합니다.
print("로그 확률:")
print(log_prob)
print("\n정답 클래스 번호:", target)
print("정답 클래스 로그 확률:", correct_log_prob)
print("데이터별 NLLLoss:", manual_nll_each)
print("직접 계산한 NLLLoss:", manual_nll.item())
print("PyTorch NLLLoss:", torch_nll.item())

## 9. KLDivLoss: KL Divergence

KL Divergence는 두 확률분포가 얼마나 다른지 측정하는 손실함수입니다.

공식은 다음과 같습니다.

$$
D_{KL}(P||Q)=\sum_i P(i)\log\frac{P(i)}{Q(i)}
$$

여기서 `P`는 실제 분포, `Q`는 예측 분포입니다.

PyTorch의 `KLDivLoss`는 보통 입력값으로 로그 확률을 받고, 타깃값으로 일반 확률분포를 받습니다.

In [ ]:
# 실제 확률분포 P를 생성합니다.
# 각 행의 합은 1이 되어야 합니다.
# 예를 들어 첫 번째 데이터는 클래스0 확률 0.7, 클래스1 확률 0.2, 클래스2 확률 0.1입니다.
target_prob = torch.tensor([
    [0.7, 0.2, 0.1],
    [0.1, 0.8, 0.1]
])

# 모델이 예측한 확률분포 Q를 생성합니다.
pred_prob = torch.tensor([
    [0.6, 0.3, 0.1],
    [0.2, 0.7, 0.1]
])

# PyTorch의 KLDivLoss는 입력값으로 로그 확률을 받기 때문에 log를 적용합니다.
log_pred_prob = torch.log(pred_prob)

# KL Divergence 공식을 직접 계산합니다.
# target_prob * log(target_prob / pred_prob)를 클래스 방향으로 모두 더합니다.
manual_kl_each = torch.sum(target_prob * torch.log(target_prob / pred_prob), dim=1)

# 데이터별 KL 값을 평균냅니다.
manual_kl = manual_kl_each.mean()

# PyTorch의 KLDivLoss 객체를 생성합니다.
# reduction='batchmean'은 배치 크기로 나누는 방식입니다.
# KL Divergence에서 수학적으로 가장 자연스러운 평균 방식으로 많이 사용됩니다.
kl_loss = nn.KLDivLoss(reduction='batchmean')

# PyTorch 내장 함수로 KLDivLoss를 계산합니다.
torch_kl = kl_loss(log_pred_prob, target_prob)

# 결과를 출력합니다.
print("실제 확률분포 P:")
print(target_prob)
print("\n예측 확률분포 Q:")
print(pred_prob)
print("\nlog(Q):")
print(log_pred_prob)
print("\n데이터별 KL Divergence:", manual_kl_each)
print("직접 계산한 KL Divergence 평균:", manual_kl.item())
print("PyTorch KLDivLoss:", torch_kl.item())

## 10. 손실함수 선택 기준 정리

| 문제 유형 | 예측 대상 | 대표 손실함수 | PyTorch 함수 |
|---|---|---|---|
| 회귀 | 연속 숫자 | MSE | `nn.MSELoss()` |
| 회귀 | 연속 숫자 | MAE | `nn.L1Loss()` |
| 회귀 | 이상치 포함 숫자 | Huber Loss | `nn.HuberLoss()` |
| 이진 분류 | 0 또는 1 | BCE | `nn.BCELoss()` |
| 이진 분류 | 0 또는 1 | Sigmoid + BCE | `nn.BCEWithLogitsLoss()` |
| 다중 분류 | 여러 클래스 중 하나 | Cross Entropy | `nn.CrossEntropyLoss()` |
| 로그 확률 분류 | 여러 클래스 중 하나 | NLL | `nn.NLLLoss()` |
| 확률분포 비교 | 분포와 분포 | KL Divergence | `nn.KLDivLoss()` |

실무에서는 다음과 같이 선택하는 경우가 많습니다.

- 숫자 예측: `MSELoss`
- 이상치가 많은 숫자 예측: `HuberLoss`
- 이진 분류: `BCEWithLogitsLoss`
- 다중 클래스 분류: `CrossEntropyLoss`
- 확률분포 비교: `KLDivLoss`

## 11. 전체 핵심 요약

손실함수는 모델이 얼마나 틀렸는지를 수치로 표현하는 함수입니다.

모델은 손실값을 줄이는 방향으로 가중치와 편향을 수정합니다.

따라서 딥러닝 모델을 설계할 때는 다음 순서로 생각해야 합니다.

1. 해결하려는 문제가 회귀인지 분류인지 확인합니다.
2. 출력층이 어떤 값을 내보내야 하는지 결정합니다.
3. 출력값의 형태에 맞는 손실함수를 선택합니다.
4. 손실값이 줄어드는지 확인하면서 모델을 학습합니다.

특히 PyTorch에서는 다음 두 가지를 기억해야 합니다.

- `CrossEntropyLoss`는 내부적으로 Softmax를 포함합니다.
- `BCEWithLogitsLoss`는 내부적으로 Sigmoid를 포함합니다.

따라서 이 두 손실함수를 사용할 때는 출력층에 Softmax나 Sigmoid를 따로 붙이지 않는 것이 일반적입니다.